## Section 2 Try it Yourself


### 🔧 Try it yourself — Libraries and loading
1. Find any shapefile or GeoJSON from your local government's open data portal (Seattle, King County, and most cities publish these for free). Download it.
2. Load it using `sf` (R) or `GeoPandas` (Python) and run the sanity-check function.
3. Does the CRS look right? Does the bounding box match where you'd expect the data to be geographically?
4. What geometry type is it? Does that match what you'd expect for this dataset?

### I'll try Oregon Metro’s Outdoor Recreation and Conservation Areas (ORCA) layer
- It covers parks, natural areas and other open space across the Portland region, published by Metro’s Data Resource Center as a public ArcGIS feature service, so no sign-in is needed. 
- A couple extra new steps here - notice the REST services accept queries and you can use direct SQL query parameters when you read in
- I'm only reading in where the 'unit type' is LIKE 'Natural Area'

🟢 **Python**

In [ ]:
import geopandas as gpd
import requests

def sanity_check(gdf, label="layer"):
    """Print key properties of a GeoDataFrame."""
    print(f"\n==== {label} ====")
    print(f"Features: {len(gdf)} | Columns: {len(gdf.columns)}")
    print(f"Geometry types: {gdf.geometry.geom_type.unique()}")
    print(f"CRS: {gdf.crs}")
    null_count = gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
    print(f"Null/empty geometries: {null_count}")

url = ("https://services2.arcgis.com/McQ0OlIABe29rJJy/ArcGIS/rest/services/"
       "Outdoor_Recreation_and_Conservation_Areas_ORCA/FeatureServer/5/query")

params = {
    "where": "UNITTYPE LIKE '%Park%' OR UNITTYPE LIKE '%Natural Area%'",
    "outFields": "*",
    "f": "geojson",
}

r = requests.get(url, params=params)
orca = gpd.GeoDataFrame.from_features(r.json()["features"], crs=4326).to_crs(2913)

sanity_check(orca, "Metro ORCA — Parks & Natural Areas")
print(orca["UNITTYPE"].value_counts())


==== Metro ORCA — Parks & Natural Areas ====
Features: 2000 | Columns: 19
Geometry types: ['Polygon' 'MultiPolygon']
CRS: EPSG:2913
Null/empty geometries: 0
UNITTYPE
Natural Area    1052
Park             948
Name: count, dtype: int64


🔵 **`R`**

In [ ]:
library(sf)
library(arcgislayers)

url  <- "https://services2.arcgis.com/McQ0OlIABe29rJJy/ArcGIS/rest/services/Outdoor_Recreation_and_Conservation_Areas_ORCA/FeatureServer/5"
orca <- arc_read(url)        # public layer, so no sign-in needed

sanity_check_r <- function(layer, label = "layer") {
  cat("\n====", label, "====\n")
  cat("Features:", nrow(layer), "| Columns:", ncol(layer), "\n")
  cat("Geometry:", as.character(unique(st_geometry_type(layer))), "\n")
  cat("CRS EPSG:", st_crs(layer)$epsg, "\n")
  cat("Empty geoms:", sum(st_is_empty(layer)), "\n")
}
sanity_check_r(orca, "Metro ORCA")
st_bbox(orca)

Linking to GEOS 3.13.1, GDAL 3.11.4, PROJ 9.7.0; sf_use_s2() is TRUE


Warning message:
package ‘sf’ was built under R version 4.5.2 
Warning message:
package ‘arcgislayers’ was built under R version 4.5.2 


                                                                     88%
==== Metro ORCA ====
Features: 15083 | Columns: 19 
Geometry: POLYGON 
CRS EPSG: 2913 
Empty geoms: 0 


     xmin      ymin      xmax      ymax 
7435739.1  447015.5 7898712.1  783256.0 

## Section 3 Try it Yourself

🟢 **Python**

In [ ]:
import sys
import os

print(f"Python version:    {sys.version.split()[0]}")
print(f"Environment path:  {sys.prefix}")
print(f"Environment name:  {os.path.basename(sys.prefix)}")

# Confirm arcpy is accessible (only works if this is a cloned ArcGIS Pro env)
try:
    import arcpy
    print(f"arcpy version:     {arcpy.GetInstallInfo()['Version']}")
    print("✓ ArcGIS Pro environment confirmed — arcpy is available.")
except ImportError:
    print("– arcpy not available in this environment (expected if not a Pro clone).")

# Confirm open-source packages
import geopandas as gpd
print(f"geopandas version: {gpd.__version__}")
print("✓ arcpy with geopandas confirmed.")

Python version:    3.13.7
Environment path:  C:\Users\mwebe\AppData\Local\ESRI\conda\envs\gispr
Environment name:  gispr
The version of the binaries in the active Python environment do
  not match the version of the binaries in the ArcGIS application.

Active environment version : 3.6
arcgispro-py3 environment version : 3.7

To update the existing environment see the conda proup subcommand 
        > conda proup --help

To manually create a new environment for use with the current ArcGIS application:
 1. Generate a list of additional packages installed in your current environment,
    with the conda command:
        > conda env export > env.yaml
 2. (Optional) If you have additional dependencies installed through pip,
    find those with pip freeze:
        > pip freeze > requirements.txt
 3. Create a new environment by cloning arcgispro-py3, and activate it:
        > conda create --clone arcgispro-py3 --name my-env --pinned
        > activate my-env
 4. Add back missing conda package

## Section 5 Try it Yourself

### 🔧 Try it yourself — Loops and conditionals
1. Add a column `area_tier` that classifies features by **geometry area** into `"large"`, `"medium"`, and `"small"`. Use `st_area()` in R (after projecting to an equal-area CRS) or `.area` in Python after `.to_crs("EPSG:6933")`.
2. Using a `for` loop, print the name of every feature where your two tier columns disagree (a large-pop country that is small-area, for example). How many are there?
3. Try the same classification using the vectorized approach (`case_when` in R or `np.select` in Python). Do the results match?

🟢 **Python**

In [ ]:
nat_earth_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"

world = (
    gpd.read_file(nat_earth_url)
      [["NAME", "CONTINENT", "POP_EST", "ISO_A3", "GDP_MD", "geometry"]]
      .rename(columns=str.lower)
      .rename(columns={"gdp_md": "gdp_md_est"})
)

import numpy as np
list(world)
# Area needs an equal-area CRS — world is EPSG:4326 (degrees)
world["area_km2"] = world.to_crs("EPSG:6933").area / 1e6   # World Cylindrical Equal Area, m² → km²

# ── 1. Classify by area with an explicit for loop ─────────────
area_tiers = []
for area in world["area_km2"]:
    if area >= 1_000_000:
        area_tiers.append("large")      # ≥ 1,000,000 km²
    elif area >= 250_000:
        area_tiers.append("medium")     # 250,000–1,000,000 km²
    else:
        area_tiers.append("small")      # < 250,000 km²
world["area_tier"] = area_tiers

print("Area tier breakdown:")
print(world["area_tier"].value_counts())

# # ── 2. Loop: where do the population and area tiers disagree? ─
conditions = [
    world["pop_est"] >= 100_000_000,
    world["pop_est"] >=  50_000_000,
    world["pop_est"] >=  10_000_000,
]
world["pop_tier"] = np.select(
    conditions,
    ["large", "medium", "small"],
    default="micro"
)
# # pop_tier has four levels; count "micro" as "small" so the scales line up
mismatches = []
for idx, row in world.iterrows():
    pop = "small" if row["pop_tier"] == "micro" else row["pop_tier"]
    if pop != row["area_tier"]:
        mismatches.append(row["name"])
        print(f"{row['name']:<28} pop: {row['pop_tier']:<7} area: {row['area_tier']}")

print(f"\n{len(mismatches)} of {len(world)} countries have different population and area tiers")

# ── 3. Vectorized version with np.select — does it match? ─────
world["area_tier_vec"] = np.select(
    [world["area_km2"] >= 1_000_000,
     world["area_km2"] >=   250_000],
    ["large", "medium"],
    default="small"
)
print("Loop and np.select agree:", (world["area_tier"] == world["area_tier_vec"]).all())

Area tier breakdown:
area_tier
small     99
medium    48
large     30
Name: count, dtype: int64
Canada                       pop: small   area: large
Kazakhstan                   pop: small   area: large
Uzbekistan                   pop: small   area: medium
Papua New Guinea             pop: micro   area: medium
Argentina                    pop: small   area: large
Chile                        pop: small   area: medium
Dem. Rep. Congo              pop: medium  area: large
Somalia                      pop: small   area: medium
Sudan                        pop: small   area: large
Chad                         pop: small   area: large
Norway                       pop: micro   area: medium
Greenland                    pop: micro   area: large
South Africa                 pop: medium  area: large
Bolivia                      pop: small   area: large
Peru                         pop: small   area: large
Colombia                     pop: medium  area: large
Venezuela                    pop: s

🔵 **`R`**

In [ ]:
# [R] Try it yourself — area tiers, tier disagreements, loop vs. vectorized
# Run the population-tier cell above first (it creates `states` with pop_tier)
library(sf)
library(spData)
library(dplyr)

states <- us_states

# ── Explicit for loop: flag states over 10 million people ─────
large_states <- c()
for (i in seq_len(nrow(states))) {
  pop <- states$total_pop_15[i]
  if (!is.na(pop) && pop > 10000000) {
    large_states <- c(large_states, states$NAME[i])
  }
}
cat("States with pop > 10M (loop approach):")
cat("\n ", paste(large_states, collapse = ", "), "\n")

# ── Vectorized alternative: dplyr::case_when (idiomatic R) ────
states <- states |>
  mutate(
    pop_tier = case_when(
      total_pop_15 >= 10000000 ~ "large",
      total_pop_15 >=  3000000 ~ "medium",
      total_pop_15 >=  1000000 ~ "small",
      TRUE                     ~ "micro"
    )
  )

# Area needs an equal-area CRS — us_states is EPSG:4269 (degrees)
states_ea <- st_transform(states, 5070)              # NAD83 / CONUS Albers, metres
states$area_km2 <- as.numeric(st_area(states_ea)) / 1e6

# ── 1. Classify by area with an explicit for loop ─────────────
states$area_tier <- NA_character_
for (i in seq_len(nrow(states))) {
  a <- states$area_km2[i]
  if (a >= 250000) {
    states$area_tier[i] <- "large"                   # ≥ 250,000 km²
  } else if (a >= 100000) {
    states$area_tier[i] <- "medium"                  # 100,000–250,000 km²
  } else {
    states$area_tier[i] <- "small"                   # < 100,000 km²
  }
}
cat("Area tier breakdown:\n")
print(table(states$area_tier))

# ── 2. Loop: where do the population and area tiers disagree? ─
# pop_tier has four levels; count "micro" as "small" so the scales line up
mismatches <- c()
for (i in seq_len(nrow(states))) {
  pop <- ifelse(states$pop_tier[i] == "micro", "small", states$pop_tier[i])
  if (pop != states$area_tier[i]) {
    mismatches <- c(mismatches, states$NAME[i])
    cat(sprintf("%-16s pop: %-7s area: %s\n",
                states$NAME[i], states$pop_tier[i], states$area_tier[i]))
  }
}
cat("\n", length(mismatches), "of", nrow(states), "states have different population and area tiers\n")

# ── 3. Vectorized version with case_when — does it match? ─────
states <- states |>
  mutate(area_tier_vec = case_when(
    area_km2 >= 250000 ~ "large",
    area_km2 >= 100000 ~ "medium",
    TRUE               ~ "small"
  ))
cat("Loop and case_when agree:", identical(states$area_tier, states$area_tier_vec), "\n")

Linking to GEOS 3.13.1, GDAL 3.11.4, PROJ 9.7.0; sf_use_s2() is TRUE


Warning message:
package ‘sf’ was built under R version 4.5.2 


To access larger datasets in this package, install the spDataLarge package with:
`install.packages('spDataLarge', repos='https://nowosad.github.io/drat/',
type='source')`


Warning message:
package ‘spData’ was built under R version 4.5.3 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Warning message:
package ‘dplyr’ was built under R version 4.5.2 


States with pop > 10M (loop approach):
  Florida, Georgia, New York, Pennsylvania, Texas, California, Illinois, Ohio 
Area tier breakdown:

 large medium  small 
     9     27     13 
Arizona          pop: medium  area: large
Colorado         pop: medium  area: large
Connecticut      pop: medium  area: small
Florida          pop: large   area: medium
Georgia          pop: large   area: medium
Idaho            pop: small   area: medium
Indiana          pop: medium  area: small
Kansas           pop: small   area: medium
Massachusetts    pop: medium  area: small
Montana          pop: small   area: large
Nevada           pop: small   area: large
New Jersey       pop: medium  area: small
New York         pop: large   area: medium
North Dakota     pop: micro   area: medium
Pennsylvania     pop: large   area: medium
South Carolina   pop: medium  area: small
South Dakota     pop: micro   area: medium
Arkansas         pop: small   area: medium
Illinois         pop: large   area: medium
Maryland

## Section 6 Try it Yourself

### 🔧 Try it yourself — Functions
1. Write a function `classify_density(density_value)` that returns `"dense"`, `"moderate"`, or `"sparse"` based on thresholds you choose. Apply it to the `pop_density` column you computed above.
2. Write a function `check_crs(layer, expected_epsg)` that returns `True` if the layer's CRS matches the expected EPSG and `False` (with a warning message) if it doesn't. Call it on two different layers.
3. Modify the `sanity_check` function to also print the first 3 column names and the min/max of the first numeric column it finds.

🟢 **Python**

### Original cell from Module 2

In [1]:
# [Python] Three reusable functions — written once, called many times

import geopandas as gpd
import pandas as pd
import re

# ── Function 1: Sanity check (reusable across any GeoDataFrame) ──
def sanity_check(gdf, label="layer"):
    """Print key properties of a GeoDataFrame. Returns gdf for chaining."""
    print(f"\n==== {label} ====")
    print(f"  Features: {len(gdf)} | Columns: {len(gdf.columns)}")
    print(f"  Geometry types: {gdf.geometry.geom_type.unique()}")
    print(f"  CRS: {gdf.crs}")
    nulls = gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
    print(f"  Null/empty geometries: {nulls}")
    return gdf

# ── Function 2: Standardize a text column ───────────────────────
def standardize_text(text):
    """Lowercase, strip whitespace, replace spaces with underscores."""
    if pd.isna(text):
        return None
    text = str(text).lower().strip()
    return re.sub(r"\s+", "_", text)

# ── Function 3: Compute population density ──────────────────────
def pop_density(gdf, pop_col, epsg_ea=6933):
    """Return population density in pop/km², using an equal-area projection."""
    projected = gdf.to_crs(epsg=epsg_ea)
    area_km2  = projected.geometry.area / 1e6
    return (gdf[pop_col] / area_km2).round(2)

# ── Apply all three functions ────────────────────────────────────
nat_earth_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"

world = (
    gpd.read_file(nat_earth_url)
      [["NAME", "CONTINENT", "POP_EST", "ISO_A3", "GDP_MD", "geometry"]]
      .rename(columns=str.lower)
      .rename(columns={"gdp_md": "gdp_md_est"})
)
sanity_check(world, "World Countries (raw)")

world["name_clean"]  = world["name"].apply(standardize_text)
world["pop_density"] = pop_density(world, "pop_est")

print("\nTop 5 countries by population density (pop/km²):")
print(
    world[["name", "pop_est", "pop_density"]]
    .sort_values("pop_density", ascending=False)
    .head(5)
    .to_string(index=False)
)
sanity_check(world, "World Countries (processed)")


==== World Countries (raw) ====
  Features: 177 | Columns: 6
  Geometry types: ['MultiPolygon' 'Polygon']
  CRS: EPSG:4326
  Null/empty geometries: 0

Top 5 countries by population density (pop/km²):
      name     pop_est  pop_density
Bangladesh 163046161.0      1218.96
 Palestine   4685306.0       930.54
    Taiwan  23568378.0       685.74
   Lebanon   6855713.0       679.58
    Rwanda  12626950.0       540.42

==== World Countries (processed) ====
  Features: 177 | Columns: 8
  Geometry types: ['MultiPolygon' 'Polygon']
  CRS: EPSG:4326
  Null/empty geometries: 0


,name,continent,pop_est,iso_a3,gdp_md_est,geometry,name_clean,pop_density
0,Fiji,Oceania,889953.0,FJI,5496,"MULTIPOLYGON (((180.00000 -16.06713, 180.00000...",fiji,46.14
1,Tanzania,Africa,58005463.0,TZA,63177,"POLYGON ((33.90371 -0.95000, 34.07262 -1.05982...",tanzania,62.19
2,W. Sahara,Africa,603253.0,ESH,907,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...",w._sahara,6.24
3,Canada,North America,37589262.0,CAN,1736425,"MULTIPOLYGON (((-122.84000 49.00000, -122.9742...",canada,3.74
4,United States of America,North America,328239523.0,USA,21433226,"MULTIPOLYGON (((-122.84000 49.00000, -120.0000...",united_states_of_america,34.52
...,...,...,...,...,...,...,...,...
172,Serbia,Europe,6944975.0,SRB,51475,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...",serbia,90.93
173,Montenegro,Europe,622137.0,MNE,5542,"POLYGON ((20.07070 42.58863, 19.80161 42.50009...",montenegro,46.27
174,Kosovo,Europe,1794248.0,-99,7926,"POLYGON ((20.59025 41.85541, 20.52295 42.21787...",kosovo,159.75
175,Trinidad and Tobago,North America,1394973.0,TTO,24269,"POLYGON ((-61.68000 10.76000, -61.10500 10.890...",trinidad_and_tobago,180.25


In [2]:
# [Python] Try it yourself — Functions (solutions)
# Run the cell above first (it creates `world` with pop_density)
import warnings
import pandas as pd
import geopandas as gpd

# ── 1. classify_density(): one value in, one label out ────────
def classify_density(density_value, dense=150, moderate=30):
    """Label a population density (people per km²) as dense, moderate, or sparse."""
    if pd.isna(density_value):
        return None
    if density_value >= dense:
        return "dense"        # ≥ 150 people per km²
    elif density_value >= moderate:
        return "moderate"     # 30–150 people per km²
    else:
        return "sparse"       # < 30 people per km²

world["density_class"] = world["pop_density"].apply(classify_density)
print(world["density_class"].value_counts())
print(world[["name", "pop_density", "density_class"]]
      .sort_values("pop_density", ascending=False)
      .head(6)
      .to_string(index=False))

# ── 2. check_crs(): True if the EPSG matches, False + warning if not ──
def check_crs(layer, expected_epsg):
    """Return True if layer's CRS is expected_epsg; otherwise warn and return False."""
    actual = layer.crs.to_epsg() if layer.crs is not None else None
    if actual == expected_epsg:
        return True
    warnings.warn(f"CRS mismatch: layer is EPSG:{actual}, expected EPSG:{expected_epsg}")
    return False

world_ea = world.to_crs(epsg=6933)          # an equal-area copy, like the one pop_density() makes
print("\nworld    is EPSG:4326?", check_crs(world, 4326))      # True
print("world_ea is EPSG:4326?", check_crs(world_ea, 4326))     # False + warning

# ── 3. sanity_check(), extended ───────────────────────────────
def sanity_check(gdf, label="layer"):
    """Print key properties of a GeoDataFrame. Returns gdf for chaining."""
    print(f"\n==== {label} ====")
    print(f"  Features: {len(gdf)} | Columns: {len(gdf.columns)}")
    print(f"  Geometry types: {gdf.geometry.geom_type.unique()}")
    print(f"  CRS: {gdf.crs}")
    nulls = gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
    print(f"  Null/empty geometries: {nulls}")

    # New: first 3 column names
    print(f"  First 3 columns: {list(gdf.columns[:3])}")

    # New: min/max of the first numeric column
    num_cols = gdf.select_dtypes(include="number").columns
    if len(num_cols) > 0:
        col = num_cols[0]
        print(f"  {col}: min = {gdf[col].min():,} | max = {gdf[col].max():,}")
    else:
        print("  No numeric columns found")
    return gdf

sanity_check(world, "World Countries")
sanity_check(world[world["continent"] == "Africa"], "Africa only")

density_class
moderate    88
sparse      51
dense       38
Name: count, dtype: int64
       name  pop_density density_class
 Bangladesh      1218.96         dense
  Palestine       930.54         dense
     Taiwan       685.74         dense
    Lebanon       679.58         dense
     Rwanda       540.42         dense
South Korea       522.10         dense

world    is EPSG:4326? True
world_ea is EPSG:4326? False

==== World Countries ====
  Features: 177 | Columns: 9
  Geometry types: ['MultiPolygon' 'Polygon']
  CRS: EPSG:4326
  Null/empty geometries: 0
  First 3 columns: ['name', 'continent', 'pop_est']
  pop_est: min = 140.0 | max = 1,397,715,000.0

==== Africa only ====
  Features: 51 | Columns: 9
  Geometry types: ['Polygon' 'MultiPolygon']
  CRS: EPSG:4326
  Null/empty geometries: 0
  First 3 columns: ['name', 'continent', 'pop_est']
  pop_est: min = 603,253.0 | max = 200,963,599.0


<positron-console-cell-2>:32: UserWarning: CRS mismatch: layer is EPSG:6933, expected EPSG:4326


,name,continent,pop_est,iso_a3,gdp_md_est,geometry,name_clean,pop_density,density_class
1,Tanzania,Africa,58005463.0,TZA,63177,"POLYGON ((33.90371 -0.95000, 34.07262 -1.05982...",tanzania,62.19,moderate
2,W. Sahara,Africa,603253.0,ESH,907,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...",w._sahara,6.24,sparse
11,Dem. Rep. Congo,Africa,86790567.0,COD,50400,"POLYGON ((29.34000 -4.49998, 29.51999 -5.41998...",dem._rep._congo,37.35,moderate
12,Somalia,Africa,10192317.3,SOM,4719,"POLYGON ((41.58513 -1.68325, 40.99300 -0.85829...",somalia,21.05,sparse
13,Kenya,Africa,52573973.0,KEN,95503,"POLYGON ((39.20222 -4.67677, 37.76690 -3.67712...",kenya,88.99,moderate
14,Sudan,Africa,42813238.0,SDN,30513,"POLYGON ((24.56737 8.22919, 23.80581 8.66632, ...",sudan,23.14,sparse
15,Chad,Africa,15946876.0,TCD,11314,"POLYGON ((23.83766 19.58047, 23.88689 15.61084...",chad,12.55,sparse
25,South Africa,Africa,58558270.0,ZAF,351431,"POLYGON ((16.34498 -28.57671, 16.82402 -28.082...",south_africa,48.14,moderate
26,Lesotho,Africa,2125268.0,LSO,2376,"POLYGON ((28.97826 -28.95560, 29.32517 -29.257...",lesotho,77.28,moderate
48,Zimbabwe,Africa,14645468.0,ZWE,21440,"POLYGON ((31.19141 -22.25151, 30.65987 -22.151...",zimbabwe,38.92,moderate


🔵 **`R`**

### Original cell from Module 2

In [ ]:
library(sf)
library(spData)
library(dplyr)

# ── Function 1: Sanity check (reusable across any sf layer) ──────
sanity_check_sf <- function(layer, label = "layer") {
  cat("\n====" , label, "====\n")
  cat("  Features:", nrow(layer), "| Cols:", ncol(layer), "\n")
  cat("  Geometry:", as.character(unique(st_geometry_type(layer))), "\n")
  cat("  CRS EPSG:", st_crs(layer)$epsg, "\n")
  cat("  Null geoms:", sum(st_is_empty(layer)), "\n")
  invisible(layer)   # return the layer so the function can be piped
}

# ── Function 2: Standardize a text attribute column ─────────────
# Useful before joins — lowercase + no extra whitespace
standardize_text <- function(x) {
  x <- trimws(tolower(as.character(x)))
  gsub("\\s+", "_", x)   # spaces to underscores
}

# ── Function 3: Compute population density ──────────────────────
# Projects to equal-area, computes area in km², returns density
pop_density <- function(layer, pop_col, crs_ea = 6933) {
  projected  <- st_transform(layer, crs = crs_ea)
  area_km2   <- as.numeric(st_area(projected)) / 1e6
  round(layer[[pop_col]] / area_km2, 2)
}

# ── Apply all three functions ────────────────────────────────────
states <- us_states
sanity_check_sf(states, "US States (raw)")

states <- states |>
  mutate(
    name_clean  = standardize_text(NAME),
    pop_density = pop_density(states, "total_pop_15")
  )

cat("\nTop 5 states by population density (per km²):\n")
top5 <- states |>
  st_drop_geometry() |>
  select(NAME, total_pop_15, pop_density) |>
  arrange(desc(pop_density)) |>
  head(5)
print(top5)

sanity_check_sf(states, "US States (processed)")


==== US States (raw) ====
  Features: 49 | Cols: 7 
  Geometry: MULTIPOLYGON 
  CRS EPSG: 4269 
  Null geoms: 0 

Top 5 states by population density (per km²):
                  NAME total_pop_15 pop_density
1 District of Columbia       647484     3633.08
2           New Jersey      8904413      439.21
3         Rhode Island      1053661      384.13
4        Massachusetts      6705586      320.76
5          Connecticut      3593222      277.16

==== US States (processed) ====
  Features: 49 | Cols: 9 
  Geometry: MULTIPOLYGON 
  CRS EPSG: 4269 
  Null geoms: 0 


### New function

In [ ]:
# [R] Try it yourself — Functions (solutions)
# Run the cell above first (it creates `states` with pop_density, and sanity_check_sf)
library(sf)
library(spData)
library(dplyr)

# ── 1. classify_density(): one value in, one label out ────────
classify_density <- function(density_value, dense = 100, moderate = 25) {
  if (is.na(density_value)) return(NA_character_)
  if (density_value >= dense) {
    "dense"            # ≥ 100 people per km²
  } else if (density_value >= moderate) {
    "moderate"         # 25–100 people per km²
  } else {
    "sparse"           # < 25 people per km²
  }
}

# The function takes a single value, so apply it element by element
states$density_class <- sapply(states$pop_density, classify_density)
print(table(states$density_class))
states |>
  st_drop_geometry() |>
  select(NAME, pop_density, density_class) |>
  arrange(desc(pop_density)) |>
  head(6) |>
  print()

# ── 2. check_crs(): TRUE if the EPSG matches, FALSE + warning if not ──
check_crs <- function(layer, expected_epsg) {
  actual <- st_crs(layer)$epsg
  if (!is.na(actual) && actual == expected_epsg) {
    return(TRUE)
  }
  warning(sprintf("CRS mismatch: layer is EPSG:%s, expected EPSG:%s",
                  ifelse(is.na(actual), "unknown", actual), expected_epsg),
          call. = FALSE)
  FALSE
}

check_crs(states, 4269)    # us_states is NAD83 (EPSG:4269)  → TRUE
check_crs(world, 4269)     # spData::world is EPSG:4326      → FALSE + warning

# ── 3. sanity_check_sf(), extended ────────────────────────────
sanity_check_sf <- function(layer, label = "layer") {
  cat("\n====", label, "====\n")
  cat("  Features:", nrow(layer), "| Cols:", ncol(layer), "\n")
  cat("  Geometry:", as.character(unique(st_geometry_type(layer))), "\n")
  cat("  CRS EPSG:", st_crs(layer)$epsg, "\n")
  cat("  Null geoms:", sum(st_is_empty(layer)), "\n")

  # New: first 3 column names
  cat("  First 3 columns:", paste(head(names(layer), 3), collapse = ", "), "\n")

  # New: min/max of the first numeric attribute column
  attrs    <- st_drop_geometry(layer)
  num_cols <- names(attrs)[sapply(attrs, is.numeric)]
  if (length(num_cols) > 0) {
    x <- as.numeric(attrs[[num_cols[1]]])   # as.numeric drops units (e.g. km²)
    cat(sprintf("  %s: min = %s | max = %s\n", num_cols[1],
                format(min(x, na.rm = TRUE), big.mark = ","),
                format(max(x, na.rm = TRUE), big.mark = ",")))
  } else {
    cat("  No numeric columns found\n")
  }
  invisible(layer)
}

sanity_check_sf(states, "US States")
sanity_check_sf(world,  "World Countries")


   dense moderate   sparse 
      11       19       19 
                  NAME pop_density density_class
1 District of Columbia     3633.08         dense
2           New Jersey      439.21         dense
3         Rhode Island      384.13         dense
4        Massachusetts      320.76         dense
5          Connecticut      277.16         dense
6             Maryland      221.10         dense


Warning message:
CRS mismatch: layer is EPSG:4326, expected EPSG:4269 



==== US States ====
  Features: 49 | Cols: 10 
  Geometry: MULTIPOLYGON 
  CRS EPSG: 4269 
  Null geoms: 0 
  First 3 columns: GEOID, NAME, REGION 
  AREA: min = 178.21 | max = 687,714.3

==== World Countries ====
  Features: 177 | Cols: 11 
  Geometry: MULTIPOLYGON 
  CRS EPSG: 4326 
  Null geoms: 0 
  First 3 columns: iso_a2, name_long, continent 
  area_km2: min = 2,416.87 | max = 17,018,507
